In [22]:
# Import Required Libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve, auc
from sklearn import preprocessing
from sklearn.datasets import load_breast_cancer
import matplotlib.pyplot as plt
import seaborn as sns
# import torch

In [23]:
# Import Breast Cancer Wisconsin Dataset
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

print("Dataset shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeature names:")
print(cancer.feature_names)
print("\nTarget names:", cancer.target_names)
#print("\nDataset description:")
#print(cancer.DESCR[:500])

Dataset shape: (569, 30)
Target shape: (569,)

Feature names:
['mean radius' 'mean texture' 'mean perimeter' 'mean area'
 'mean smoothness' 'mean compactness' 'mean concavity'
 'mean concave points' 'mean symmetry' 'mean fractal dimension'
 'radius error' 'texture error' 'perimeter error' 'area error'
 'smoothness error' 'compactness error' 'concavity error'
 'concave points error' 'symmetry error' 'fractal dimension error'
 'worst radius' 'worst texture' 'worst perimeter' 'worst area'
 'worst smoothness' 'worst compactness' 'worst concavity'
 'worst concave points' 'worst symmetry' 'worst fractal dimension']

Target names: ['malignant' 'benign']


### 📊 Good Training, Cross‑Validation, and Test Set Split

A common and effective split for machine learning datasets is **70% training, 15% validation, 15% test**:

- **Training set (70%)**: Used to train the model parameters. This is the largest portion to capture the underlying patterns.
- **Validation set (15%)**: Used for hyperparameter tuning and model selection during cross‑validation. Helps prevent overfitting by evaluating performance on unseen data during training.
- **Test set (15%)**: Held out entirely until the end for final evaluation. Provides an unbiased estimate of real‑world performance.

This split balances having enough data for training while reserving sufficient samples for validation and testing. For smaller datasets, consider 80/10/10 or use k‑fold cross‑validation on the training set (e.g., 5‑fold CV) with a separate test set. Always stratify splits for imbalanced classes to maintain proportions.

In scikit‑learn you can use `cross_val_score` or `GridSearchCV` with `LogisticRegression` to perform cross‑validation seamlessly.

### ⚙️ Setting Model Parameters for Logistic Regression

In scikit‑learn's `LogisticRegression`, key parameters to tune for better performance:

- **`penalty`**: Regularization type ('l1', 'l2', 'elasticnet', or 'none'). 'l2' is default for ridge regression; 'l1' for lasso.
- **`C`**: Inverse regularization strength (smaller values = stronger regularization). Default is 1.0; try values like 0.01, 0.1, 1, 10, 100.
- **`solver`**: Optimization algorithm ('newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga'). Choose based on dataset size and penalty (e.g., 'liblinear' for small datasets with l1).
- **`max_iter`**: Maximum iterations for convergence. Increase if the model doesn't converge (default 100).
- **`random_state`**: Set for reproducible results.

Example instantiation:  
```python
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(penalty='l2', C=1.0, solver='lbfgs', max_iter=1000, random_state=42)
```

Use `GridSearchCV` or `RandomizedSearchCV` with cross‑validation to find optimal parameters automatically.

### 🔧 Optimization Algorithms in Logistic Regression

Scikit‑learn's `LogisticRegression` supports several solvers for minimizing the loss function. Here's a quick overview:

- **`newton-cg`**: Newton‑Conjugate Gradient. Uses Newton's method with conjugate gradient for large datasets. Good for l2/elasticnet penalties. Handles multinomial loss.
- **`lbfgs`**: Limited‑memory BFGS. Quasi‑Newton method, memory‑efficient for large problems. Default for l2/elasticnet. Supports multinomial loss.
- **`liblinear`**: Coordinate descent for small‑to‑medium datasets. Supports l1/l2 penalties but not multinomial. Fast for sparse data.
- **`sag`**: Stochastic Average Gradient. Good for large datasets with l2 penalty. Converges faster than SGD but slower than lbfgs for small data.
- **`saga`**: Improved SAG with l1 regularization support. Handles large datasets and sparse features well. Supports elasticnet.

Choose based on dataset size, penalty type, and whether multinomial classification is needed. For small datasets, `liblinear` or `lbfgs`; for large ones, `sag` or `saga`.

In [27]:
X_train, X_cross_val_test, y_train, y_cross_val_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_cross_val, X_test, y_cross_val, y_test = train_test_split(X_cross_val_test, y_cross_val_test, test_size=0.5, random_state=42)

#Scale
scaler = preprocessing.StandardScaler().fit(X_train)
scaler.mean_
scaler.scale_
X_train_scaled = scaler.transform(X_train)

# Train logistic regression with different model parameters
pen=[None, 'l1', 'l2', 'elasticnet']
c=[0.01, 0.1, 1, 10, 100]
solv=['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']

max_iter=10000
random_state=42

highest_auc = 0
highest_f1 = 0
best_params = {}

for p in pen:
    for C in c:
        for s in solv:
            # skip combinations that the solver doesn't support
            if (s in ['newton-cg', 'lbfgs'] and p in ['l1', 'elasticnet']):
                # newton-cg & lbfgs don't support l1/elasticnet
                print(f"Skipping incompatible combination: solver '{s}' does not support penalty '{p}'.")
                continue
            if s == 'liblinear' and p not in ['l1', 'l2']:
                # liblinear only supports l1 or l2
                print(f"Skipping incompatible combination: solver '{s}' does not support penalty '{p}'.")
                continue
            if s == 'sag' and p not in [None, 'l2']:
                # sag only supports l2 or no penalty
                print(f"Skipping incompatible combination: solver '{s}' does not support penalty '{p}'.")
                continue
            # saga supports all penalties including None
            print(f"Training Logistic Regression with penalty={p}, C={C}, solver={s}")
            # supply l1_ratio for elasticnet
            if p == 'elasticnet':
                log_reg = LogisticRegression(max_iter=max_iter, random_state=random_state,
                                             penalty=p, C=C, solver=s, l1_ratio=0.5)
            else:
                log_reg = LogisticRegression(max_iter=max_iter, random_state=random_state,
                                             penalty=p, C=C, solver=s)
            log_reg.fit(X_train_scaled, y_train)

            X_cv_scaled = scaler.transform(X_cross_val)
            y_pred_prob = log_reg.predict_proba(X_cv_scaled)[:, 1]
            y_pred = log_reg.predict(X_cv_scaled)

            precision = precision_score(y_pred,y_cross_val)
            recall = recall_score(y_pred,y_cross_val)
            f1 = f1_score(y_pred,y_cross_val)

            fpr, tpr, thresholds = roc_curve(y_cross_val, y_pred_prob)
            auc_score = auc(fpr, tpr)

            if (auc_score > highest_auc) or (auc_score == highest_auc and f1 > highest_f1):
                highest_auc = auc_score
                highest_f1 = f1
                best_params = {'penalty': p, 'C': C, 'solver': s, 'l1_ratio': (0.5 if p=='elasticnet' else None)}
print(f"\nBest parameters: {best_params}, Highest AUC: {highest_auc:.4f}, Highest F1 Score: {highest_f1:.4f}")

Training Logistic Regression with penalty=None, C=0.01, solver=newton-cg
Training Logistic Regression with penalty=None, C=0.01, solver=lbfgs
Skipping incompatible combination: solver 'liblinear' does not support penalty 'None'.
Training Logistic Regression with penalty=None, C=0.01, solver=sag


C:\Users\sanna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\sklearn\linear_model\_logistic.py:1207: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
C:\Users\sanna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\sklearn\linear_model\_logistic.py:1207: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
C:\Users\sanna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\sklearn\linear_model\_logistic.py:1207: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(


Training Logistic Regression with penalty=None, C=0.01, solver=saga


C:\Users\sanna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\sklearn\linear_model\_logistic.py:1207: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(


Training Logistic Regression with penalty=None, C=0.1, solver=newton-cg
Training Logistic Regression with penalty=None, C=0.1, solver=lbfgs
Skipping incompatible combination: solver 'liblinear' does not support penalty 'None'.
Training Logistic Regression with penalty=None, C=0.1, solver=sag


...

Training Logistic Regression with penalty=None, C=100, solver=newton-cg
Training Logistic Regression with penalty=None, C=100, solver=lbfgs
Skipping incompatible combination: solver 'liblinear' does not support penalty 'None'.
Training Logistic Regression with penalty=None, C=100, solver=sag


C:\Users\sanna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\sklearn\linear_model\_logistic.py:1207: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
C:\Users\sanna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\sklearn\linear_model\_logistic.py:1207: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
C:\Users\sanna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\sklearn\linear_model\_logistic.py:1207: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(


Training Logistic Regression with penalty=None, C=100, solver=saga


C:\Users\sanna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\sklearn\linear_model\_logistic.py:1207: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(


Skipping incompatible combination: solver 'newton-cg' does not support penalty 'l1'.
Skipping incompatible combination: solver 'lbfgs' does not support penalty 'l1'.
Training Logistic Regression with penalty=l1, C=0.01, solver=liblinear
Skipping incompatible combination: solver 'sag' does not support penalty 'l1'.
Training Logistic Regression with penalty=l1, C=0.01, solver=saga
Skipping incompatible combination: solver 'newton-cg' does not support penalty 'l1'.
Skipping incompatible combination: solver 'lbfgs' does not support penalty 'l1'.
Training Logistic Regression with penalty=l1, C=0.1, solver=liblinear
Skipping incompatible combination: solver 'sag' does not support penalty 'l1'.
Training Logistic Regression with penalty=l1, C=0.1, solver=saga
Skipping incompatible combination: solver 'newton-cg' does not support penalty 'l1'.
Skipping incompatible combination: solver 'lbfgs' does not support penalty 'l1'.
Training Logistic Regression with penalty=l1, C=1, solver=liblinear
Skip

### 🔄 What Are Pipelines? 

A **pipeline** in scikit‑learn bundles multiple preprocessing steps and an estimator into a single object.  
Benefits:

- Ensures the same transformations are applied during training and prediction.  
- Simplifies workflow by chaining operations.  
- Makes hyperparameter tuning easier (e.g. use `GridSearchCV` over pipeline parameters).

**Example:**
```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression())
])

# fit on training data
pipe.fit(X_train, y_train)

# predict on new samples
preds = pipe.predict(X_test)
```

You can also include feature selection, imputation, or any custom transformer in the chain.

### 🔍 GridSearchCV with Pipelines (Updated with Larger Grid)

**GridSearchCV** automates hyperparameter tuning by exhaustively searching over a specified parameter grid.  
When combined with pipelines, it tunes parameters across multiple steps (e.g., preprocessing and model hyperparameters) while ensuring consistent data transformations.

**Key Benefits:**
- Prevents data leakage by applying the same preprocessing to each CV fold.
- Simplifies parameter naming: use `<step_name>__<param_name>` (double underscore).

**What is 5-fold CV?**  
5-fold cross-validation splits the training data into 5 equal parts (folds). The model is trained on 4 folds and validated on the remaining 1, repeating this process 5 times (each fold serves as validation once). The average performance across folds provides a robust estimate, reducing overfitting risk.

**Example with Expanded Parameter Grid:**
```python
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Define pipeline
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Expanded parameter grid (note the double underscores)
param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10, 100],
    'classifier__penalty': ['l1', 'l2', 'elasticnet'],
    'classifier__solver': ['liblinear', 'saga'],
    'classifier__l1_ratio': [0.1, 0.5, 0.9]  # Only for elasticnet
}

# Grid search with 5-fold CV
grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

# Best parameters and score
print("Best params:", grid.best_params_)
print("Best CV score:", grid.best_score_)

# Predict with best model
preds = grid.predict(X_test)
```

This expanded grid tests more combinations, but be mindful of computational cost—larger grids take longer to run.

### 🚨 Data Leakage Examples

**Data leakage** occurs when information from outside the training set "leaks" into the model, leading to overly optimistic performance estimates and poor generalization. Here's how to avoid it:

1. **Scaling Before Splitting:**  
   ❌ Wrong: Scale the entire dataset first, then split.  
   ```python
   scaler = StandardScaler().fit(X)  # Uses test data info!
   X_scaled = scaler.transform(X)
   X_train, X_test = train_test_split(X_scaled, test_size=0.2)
   ```  
   ✅ Correct: Scale only on training data.  
   ```python
   X_train, X_test = train_test_split(X, test_size=0.2)
   scaler = StandardScaler().fit(X_train)
   X_train_scaled = scaler.transform(X_train)
   X_test_scaled = scaler.transform(X_test)
   ```

2. **Feature Selection on Full Dataset:**  
   ❌ Wrong: Select features using the entire dataset.  
   ```python
   selector = SelectKBest().fit(X, y)  # Test labels influence selection!
   X_selected = selector.transform(X)
   ```  
   ✅ Correct: Use only training data for selection.  
   ```python
   X_train, X_test = train_test_split(X, y, test_size=0.2)
   selector = SelectKBest().fit(X_train, y_train)
   X_train_selected = selector.transform(X_train)
   X_test_selected = selector.transform(X_test)
   ```

3. **Time-Series Data:**  
   ❌ Wrong: Train on future data to predict past.  
   ✅ Correct: Split chronologically (e.g., train on 2010-2018, test on 2019).

4. **Imputing Missing Values:**  
   ❌ Wrong: Impute using global statistics from full dataset.  
   ✅ Correct: Impute using training set statistics only.

**Prevention Tip:** Always use pipelines to ensure transformations are fitted only on training data and applied consistently.